# First ML Model: Classifying Wine Cultivars

**Week 2 Task — First ML Model**

**Dataset:** [Wine Recognition Dataset](https://archive.ics.uci.edu/dataset/109/wine) (UCI), same dataset used in Week 1's EDA — chemical measurements of 178 wines from 3 cultivars.

**Goal:** Train a first predictive model and learn to judge whether it's any good — by comparing two algorithms rather than tuning just one.

**Task type:** Classification (predict which of the 3 cultivars a wine belongs to, based on its 13 chemical properties).

## 1. Load Data & Train/Test Split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

data = load_wine(as_frame=True)
df = data.frame

X = df.drop(columns='target')
y = df['target']

print("Features shape:", X.shape)
print("Class distribution:\n", y.value_counts())

In [ ]:
# Always split into train and test BEFORE fitting anything, to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

`stratify=y` keeps the class proportions consistent between train and test, which matters here since the three cultivars aren't perfectly balanced (59 / 71 / 48 samples).

## 2. Preprocessing

From Week 1's EDA, features are on very different scales (e.g. `proline` up to ~1680 vs `hue` under 2). This matters a lot for **Logistic Regression** (a distance/gradient-based model) but not for **Random Forest** (a tree-based model splits on raw thresholds, so scaling doesn't change its behaviour). Scale for the model that needs it, and fit the scaler on the training data only.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Model 1 — Logistic Regression

A simple, interpretable baseline. Using the scaled features.

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"Logistic Regression accuracy: {acc_lr:.4f}")
print("\nClassification report:\n", classification_report(y_test, y_pred_lr))

## 4. Model 2 — Random Forest

A less interpretable but often more powerful model, and a good second algorithm to compare against since it works on unscaled data and can capture non-linear relationships.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest accuracy: {acc_rf:.4f}")
print("\nClassification report:\n", classification_report(y_test, y_pred_rf))

## 5. Comparing the Two Models

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [acc_lr, acc_rf]
})
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Logistic Regression (acc={acc_lr:.3f})")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=axes[1], colorbar=False, cmap="Greens")
axes[1].set_title(f"Random Forest (acc={acc_rf:.3f})")
plt.tight_layout()
plt.show()

## 6. What's Driving Random Forest's Predictions?

Since Random Forest performed best, check which features it actually relied on — this also cross-checks against Week 1's EDA takeaways.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, palette="viridis", legend=False)
plt.title("Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

importances.head()

## 7. Report: Metrics & Takeaways

| Model | Accuracy | Notes |
|---|---|---|
| Logistic Regression | ~0.97 | One misclassification, on an already close pair of classes. Needed scaling to perform well. |
| Random Forest | ~1.00 | Perfect classification on this test set; no scaling required. |

**Key takeaways:**

1. **Both models did well** on this dataset — expected, since Week 1's EDA already showed that several feature pairs (e.g. `total_phenols` vs `flavanoids`, `od280/od315` vs `proline`) visually separated the three classes quite cleanly. A clean, well-separated dataset like this is a best-case scenario for classification.
2. **Random Forest outperformed Logistic Regression** here, achieving perfect accuracy on the held-out test set. With only 178 total samples and a 36-sample test set, this is a small sample size — a single misclassification would already show up as a ~3% accuracy drop, so this result should be read cautiously rather than as "Random Forest is definitively superior."
3. **Random Forest's feature importances broadly agree with the EDA**: `color_intensity`, `flavanoids`, and `proline` rank among the top predictors — the same features flagged as informative or outlier-prone during Week 1's exploration.
4. **Comparing two algorithms taught more than tuning one would have**: it surfaced that scaling matters for Logistic Regression but not Random Forest, and it gave a sanity check (two different model types agreeing on which features matter) rather than trusting a single model's assumptions.
5. **Caveat:** with such a small dataset, results should ideally be validated with cross-validation (e.g. k-fold) rather than a single train/test split, to check these accuracy numbers aren't just a favorable split. That's a natural next step beyond this task.